## 0. Verificar GPU

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sin GPU: el entrenamiento puede tardar 3-4 horas.')
    print('   Considera cargar los pesos pre-entrenados desde el Release.')

## 1. Instalación de Dependencias

In [ ]:
# ─── FIX DE COMPATIBILIDAD (NumPy 2.x / OpenCV) ───────────────────────────
# Colab 2024+ usa Python 3.12 y NumPy 2.x; ultralytics antiguo necesita 1.x
# Instalamos versiones compatibles y reiniciamos el kernel automáticamente.

import subprocess, sys

# 1) Instalar paquetes con versiones fijas
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.3.40',          # compatible con Python 3.12
    'roboflow==1.1.48',
    'numpy>=1.26,<2.0',             # mantener NumPy 1.x para cv2
    'opencv-python-headless>=4.9',  # versión reciente no yanked
], check=True)

# 2) Reiniciar el kernel para que los nuevos paquetes queden activos
print('✅ Instalación completa. Reiniciando kernel...')
import os
os.kill(os.getpid(), 9)  # Colab reinicia y continúa desde la siguiente celda


In [ ]:
# ─── VERIFICACIÓN POST-REINICIO ─────────────────────────────────────────────
# Esta celda corre DESPUÉS del reinicio automático.
# Si ves errores aquí, ejecuta: Entorno de ejecución → Reiniciar y ejecutar todo

import torch, cv2, ultralytics
import numpy as np

print(f'Python     : {__import__("sys").version.split()[0]}')
print(f'NumPy      : {np.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'ultralytics: {ultralytics.__version__}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA OK    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')

# Validar que NumPy es 1.x (necesario para cv2 en este entorno)
assert int(np.__version__.split('.')[0]) < 2, 'ERROR: NumPy 2.x detectado — pip install numpy<2 y reinicia'
print('\n✅ Entorno listo para entrenamiento.')


## 2. Descarga del Dataset desde Roboflow

In [ ]:
# ⚠️ Reemplaza con tu API key de Roboflow (gratuita en roboflow.com)
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # <--- EDITAR

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
dataset = project.version(1).download('yolov8')

print(f'Dataset descargado en: {dataset.location}')
print(f'data.yaml: {dataset.location}/data.yaml')

In [ ]:
# Verificar estructura del dataset
import os, yaml

data_yaml = f'{dataset.location}/data.yaml'
with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

print('=== Configuración del dataset ===')
print(f'Clases ({len(config["names"])}): {config["names"]}')
print(f'Train : {config.get("train", "N/A")}')
print(f'Val   : {config.get("val", "N/A")}')

# Contar imágenes
for split in ['train', 'valid']:
    img_dir = f'{dataset.location}/{split}/images'
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f'Imágenes {split}: {n}')

## 3. Entrenamiento YOLOv8s

In [ ]:
from ultralytics import YOLO

# Cargar modelo base pre-entrenado en COCO
model = YOLO('yolov8s.pt')

# Entrenamiento
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='ppe_yolov8s',
    project='runs/detect',
    optimizer='SGD',
    lr0=0.01,
    patience=10,       # Early stopping si no mejora en 10 épocas
    save=True,
    plots=True,        # Guardar curvas automáticamente
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=42,           # Reproducibilidad
    verbose=True
)

print('✅ Entrenamiento completado.')
print(f'Mejor mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A"):.4f}')

## 4. Visualizar Curvas de Entrenamiento

In [ ]:
from IPython.display import Image, display
import glob

train_dir = 'runs/detect/ppe_yolov8s'

print('=== Curvas de entrenamiento ===')
for img_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    img_path = f'{train_dir}/{img_name}'
    if os.path.exists(img_path):
        print(f'\n📈 {img_name}')
        display(Image(img_path, width=800))
    else:
        print(f'⚠️  No encontrado: {img_path}')

## 5. Evaluación en Conjunto de Validación

In [ ]:
# Cargar mejor modelo entrenado
best_model = YOLO(f'runs/detect/ppe_yolov8s/weights/best.pt')

# Evaluación completa
metrics = best_model.val(
    data=f'{dataset.location}/data.yaml',
    imgsz=640,
    batch=16,
    conf=0.50,
    iou=0.50,
    plots=True,
    save_json=True,
    name='ppe_val',
    project='runs/detect'
)

print('\n=== TABLA DE MÉTRICAS ===')
print(f'{"Métrica":<20} {"Valor":>10}')
print('-' * 32)
print(f'{"Precision":<20} {metrics.box.mp:>10.4f}')
print(f'{"Recall":<20} {metrics.box.mr:>10.4f}')
print(f'{"mAP50":<20} {metrics.box.map50:>10.4f}')
print(f'{"mAP50-95":<20} {metrics.box.map:>10.4f}')
print('=' * 32)

# Por clase
print('\n=== POR CLASE ===')
names = best_model.names
print(f'{"Clase":<20} {"AP50":>8}')
print('-' * 30)
for i, ap in enumerate(metrics.box.ap50):
    print(f'{names[i]:<20} {ap:>8.4f}')

## 6. Inferencia en Imágenes de Validación

In [ ]:
import random

# Seleccionar 10 imágenes aleatorias de validación
val_images = glob.glob(f'{dataset.location}/valid/images/*.jpg')
sample_images = random.sample(val_images, min(10, len(val_images)))

# Ejecutar predicción
pred_results = best_model.predict(
    source=sample_images,
    conf=0.40,
    iou=0.50,
    save=True,
    save_conf=True,
    project='runs/detect',
    name='val_predictions'
)

print(f'✅ Predicciones guardadas en: runs/detect/val_predictions/')

# Mostrar primeras 5
print('\n=== Mostrando 5 predicciones de validación ===')
pred_dir = 'runs/detect/val_predictions'
pred_files = glob.glob(f'{pred_dir}/*.jpg')[:5]
for pf in pred_files:
    display(Image(pf, width=600))

## 7. Inferencia en Imágenes Nuevas (Fuera del Dataset)

In [ ]:
import requests, os
from PIL import Image
from io import BytesIO
from IPython.display import display
import glob

# Descargar imágenes de construcción con dominio público (Wikimedia Commons)
image_urls = [
    ('img_new_1.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/9/9c/Golden_Gate_Bridge_construction_1935.jpg/640px-Golden_Gate_Bridge_construction_1935.jpg'),
    ('img_new_2.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/81/Graz_Hauptbahnhof_-_Baustelle_2012-10-26-_-_Arbeiter.jpg/640px-Graz_Hauptbahnhof_-_Baustelle_2012-10-26-_-_Arbeiter.jpg'),
    ('img_new_3.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/17/ConstructionSite_Osaka2008.jpg/640px-ConstructionSite_Osaka2008.jpg'),
    ('img_new_4.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/320px-Good_Food_Display_-_NCI_Visuals_Online.jpg'),
    ('img_new_5.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'),
]

# Reemplazar con imágenes reales de obra
image_urls = [
    ('img_new_1.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/9/9c/Golden_Gate_Bridge_construction_1935.jpg/640px-Golden_Gate_Bridge_construction_1935.jpg'),
    ('img_new_2.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/81/Graz_Hauptbahnhof_-_Baustelle_2012-10-26-_-_Arbeiter.jpg/640px-Graz_Hauptbahnhof_-_Baustelle_2012-10-26-_-_Arbeiter.jpg'),
    ('img_new_3.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/17/ConstructionSite_Osaka2008.jpg/640px-ConstructionSite_Osaka2008.jpg'),
]

os.makedirs('new_images_input', exist_ok=True)
downloaded = []

for fname, url in image_urls:
    try:
        r = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        # Validar que es imagen real antes de guardar
        img = Image.open(BytesIO(r.content))
        img = img.convert('RGB')  # normalizar a RGB
        save_path = f'new_images_input/{fname}'
        img.save(save_path)
        downloaded.append(save_path)
        print(f'✅ {fname}: {img.size}')
    except Exception as e:
        print(f'⚠️  {fname}: {e}')

print(f'\nImágenes descargadas: {len(downloaded)}')

if not downloaded:
    print('⚠️  Sin imágenes descargadas. Usando imágenes del conjunto de validación como alternativa.')
    downloaded = glob.glob(f'{dataset.location}/valid/images/*.jpg')[:3]
    print(f'Usando {len(downloaded)} imágenes de validación como imágenes nuevas.')

# Inferencia
print('\nEjecutando inferencia...')
new_results = best_model.predict(
    source=downloaded,
    conf=0.40,
    save=True,
    project='runs/detect',
    name='new_images_predictions'
)

print('\n=== Resultados en imágenes nuevas ===')
for i, r in enumerate(new_results):
    dets = [(best_model.names[int(b.cls)], f'{float(b.conf):.2f}') for b in r.boxes] if r.boxes else []
    print(f'Imagen {i+1}: {dets if dets else "Sin detecciones"}')

# Mostrar
pred_files = glob.glob('runs/detect/new_images_predictions*/*.jpg')[:5]
for pf in pred_files:
    display(Image.open(pf))


## 8. Guardar Artefactos

In [ ]:
import shutil, os
from google.colab import files

# Comprimir resultados para descarga
shutil.make_archive('ppe_results', 'zip', 'runs/detect/ppe_yolov8s')

print('Archivos disponibles para descarga:')
print('  - ppe_results.zip (curvas, métricas, pesos)')
print('\nPeso del mejor modelo:')
best_pt = 'runs/detect/ppe_yolov8s/weights/best.pt'
if os.path.exists(best_pt):
    size_mb = os.path.getsize(best_pt) / 1e6
    print(f'  best.pt: {size_mb:.1f} MB')

# Descargar el archivo ZIP
# files.download('ppe_results.zip')  # Descomenta para descargar

print('\n✅ Pipeline completado exitosamente.')
print('Recuerda subir best.pt a GitHub Releases para reproducibilidad.')